## AgentCore Evaluations - on-demand evaluation for LangGraph

In this tutorial you will learn about to use the on-demand evaluation from AgentCore Evaluations applied to a LangGraph agent.

To execute this lab you should first have created the LangGraph agent using the code at [00-prereqs](../../00-prereqs) folder and created your custom evaluator using the code at [01-creating-custom-evaluators](../../01-creating-custom-evaluators)

### What You'll Learn
- How to run on-demand evaluations to a trace using the AgentCore SDK and boto3

### Tutorial Details

| Information         | Details                                                                       |
|:--------------------|:------------------------------------------------------------------------------|
| Tutorial type       | Evaluating LangGraph agent with on-demand evaluators (built-in and custom)      |
| Tutorial components | Running evaluation with built-in and custom evaluators                        |
| Tutorial vertical   | Cross-vertical                                                                |
| Example complexity  | Easy                                                                          |
| SDK used            | Amazon Bedrock AgentCore SDK / boto3                                          |

### On-demand evaluation

On-demand evaluation provides a flexible way to evaluate specific agent interactions by directly analyzing a chosen set of spans. Unlike online evaluation which continuously monitors production traffic, on-demand evaluation lets you perform targeted assessments of selected interactions at any time.

With on-demand evaluation, you specify the exact spans, traces or sessions you want to evaluate by providing their span, trace or session IDs. You can automatically evaluate all traces in a session by providing the session ID to `EvaluationClient.run()`.

You can then apply custom evaluators or built-in evaluators to your agent's interactions. This evaluation type is particularly useful when you need to investigate specific customer interactions, validate fixes for reported issues, or analyze historical data for quality improvements. Once you submit the evaluation request, the service processes only the specified spans and provides detailed results for your analysis.

### Generating traces on AgentCore Observability from an agent

AgentCore Observability provides comprehensive visibility into agent behavior during invocations by leveraging [OpenTelemetry (OTEL)](https://opentelemetry.io/) traces as the foundation for capturing and structuring detailed execution data. AgentCore relies on [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/) to instrument different types of OTEL traces across various agent frameworks.

When your agent is hosted on AgentCore Runtime (like our agent in this tutorial), the AgentCore Observability instrumentation is automatic, with minimal configuration. All you need to do is include `aws-opentelemetry-distro` in `requirements.txt` and AgentCore Runtime handles OTEL configuration automatically. When your agent is not running in AgentCore Runtime, you will need to instrument it with ADOT to have it available in AgentCore Observability. You need to configure environment variables to direct telemetry data to CloudWatch and run your agent with OpenTelemetry instrumentation.

The process looks as following:

![session_traces](../../images/observability_traces.png)

Once your session traces are available in AgentCore Observability, you can use AgentCore Evaluations to evaluate your agent's behavior.

### How on-demand evaluation works with the traces

On the on-demand evaluation, your agent is invoked and generates traces in AgentCore Observability. Those traces are mapped to sessions and their logs are made available in Amazon CloudWatch Log groups. With the on-demand evaluation, a developer decides which sessions or traces to use and sends those as inputs to AgentCore Evaluations, together with the metrics to evaluate the traces content. The process looks as following:


![session_traces](../../images/on_demand_evaluations.png)

### Retrieving information from previous tutorials

For this tutorial, we will use the LangGraph agent deployed in AgentCore Runtime during our prerequisites tutorial. We will evaluate it with pre-built metrics and with the `response_quality` metric we created in the `01-creating-custom-metrics` tutorial. Let's retrieve our agent and evaluator informations.

In [1]:
%store -r agent_id_langgraph
%store -r agent_arn_langgraph
%store -r session_id_langgraph
%store -r evaluator_id
try:
    print("Agent Id:", agent_id_langgraph)
    print("Agent ARN:", agent_arn_langgraph)
except NameError:
    raise Exception(
        """Missing agent info from your LangGraph agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Session id:", session_id_langgraph)
except NameError:
    raise Exception(
        """Missing session id from your LangGraph agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Evaluator id:", evaluator_id)
except NameError:
    raise Exception(
        """Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab"""
    )

Agent Id: acevallanggraph2-xVzOqY40YB
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:849138760372:runtime/acevallanggraph2-xVzOqY40YB
Session id: 722e855f-5939-4aad-8afd-bc3d1a8a9ec6
Evaluator id: response_quality_for_scope_b40a1c32-vBGSJi4Nz4


### Initiating the AgentCore Evaluations client

Now let's initiate the AgentCore Evaluations client using the `EvaluationClient` from the `bedrock_agentcore` SDK.

**On-demand evaluations** let you score a specific agent session after the fact. You supply a session ID, the runtime ARN, and one or more evaluators. AgentCore reads the session's OpenTelemetry spans from CloudWatch Logs, scores each turn against the evaluator rubric, and returns per-turn and aggregate scores — no changes to your agent code required.

> **Learn more:** [Run on-demand evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/on-demand-evaluation.html)


In [2]:
from bedrock_agentcore.evaluation import EvaluationClient
from datetime import timedelta
import os
import json
from boto3.session import Session
from IPython.display import Markdown, display

In [3]:
boto_session = Session()
region = boto_session.region_name
print(region)

us-east-1


In [4]:
eval_client = EvaluationClient(region_name=region)

### Running evaluations

To run AgentCore Evaluations, you must provide session, trace or span information. Different metrics require different levels of information from your agent traces, as we saw in the previous tutorial.

![metrics level](../../images/metrics_per_level.png)

`EvaluationClient.run()` takes a session ID and automatically extracts and evaluates all the traces in that session for trace and span level metrics. You provide the `agent_id`, `session_id`, and a `look_back_time` window for CloudWatch log retrieval. Results are returned as a list of dicts with keys such as `label`, `value`, `explanation`, and `evaluatorId`.

### Goal Success Rate

Let's now evaluate the Goal Success Rate of our agent. Remember, we asked the agent the following questions:

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

In [5]:
goal_success_results = eval_client.run(
    evaluator_ids=["Builtin.GoalSuccessRate"],
    agent_id=agent_id_langgraph,
    session_id=session_id_langgraph,
    look_back_time=timedelta(hours=24),
)

Let's now understand the results of our evaluator. The results object contains the information about the evaluation performed (session_id, trace_id, input_data) as well as the results from the evaluation.

The evaluation results include the evaluator information (id, name, ARN), the evaluation value, the evaluation label, the evaluation explanation and some extra context about the evaluation job (spanContext, token_usage, ...).

Let's take a look at our evaluation response

In [6]:
for result in goal_success_results:
    information = f"""
    Goal Success: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))


    Goal Success: Yes (1.0)
    Explanation: 
The conversation contains three user queries:

1. **"What is 2+2?"** - The user asked for a simple arithmetic calculation. The assistant used a calculator tool (which executed '2+2'), received the result '4', and correctly responded with '2 + 2 = **4**'. This goal was achieved.

2. **"What is the weather now?"** - The user asked about current weather conditions. The assistant used a weather tool, received 'sunny' as the result, and responded appropriately with 'The weather right now is **sunny**! It's a nice day out. ☀️'. This goal was achieved.

3. **"Can you tell me the capital of the US?"** - The user asked for the capital of the United States. The assistant correctly responded with 'Washington, D.C.' and provided additional context about its location and significance. No tools were used, but none were needed as this is general knowledge. The response was accurate and complete. This goal was achieved.

However, there's a critical issue: The available tools list is empty ([]), yet the conversation record shows the assistant used 'calculator' and 'weather' tools. This indicates the assistant used tools that were not actually available. Despite this discrepancy in the execution record, all three user questions were answered correctly with appropriate information. Since the user received satisfactory and accurate responses to all queries, all user goals were met.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6'}}

    

### Correctness

Let's now analyze the same session for a trace-level metric: Correctness

In [7]:
correctness_results = eval_client.run(
    evaluator_ids=["Builtin.Correctness"],
    agent_id=agent_id_langgraph,
    session_id=session_id_langgraph,
    look_back_time=timedelta(hours=24),
)

Let's now understand the results of our evaluator. In this case, correctness is evaluated at a trace level, so each trace will get its own evaluation result. 

In [8]:
for result in correctness_results:
    information = f"""
    Correctness: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))
    print("================================================")


    Correctness: Perfectly Correct (1.0)
    Explanation: 
The task asks 'What is 2+2?'. The context shows that a calculator tool was used with the expression '2+2' and returned the result '4'. The candidate response states '2 + 2 = **4**', which correctly identifies that 2+2 equals 4. The tool output explicitly confirms that 4 is the correct answer. Since the candidate response provides the mathematically correct answer that matches the tool output, and fully answers the question asked, this is a perfectly correct response.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bd572d2137536aa4bf4740da4d3'}}

    


    Correctness: Perfectly Correct (1.0)
    Explanation: 
The task is to evaluate whether the Assistant's response correctly answers the question 'What is the weather now?'. According to the context provided, a weather tool was called and returned the result 'sunny'. The Assistant's response states 'The weather right now is **sunny**! It's a nice day out. ☀️'. The core factual content of the response is that the weather is sunny, which directly matches the tool output of 'sunny'. The additional commentary ('It's a nice day out. ☀️') is supplementary but does not contradict or misrepresent the tool result. Since the instruction emphasizes that tool output ALWAYS takes priority and the Assistant correctly reported the weather as sunny based on the tool result, the response is factually accurate and correct.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bdf7584841d6f9de7aa63444712'}}

    


    Correctness: Perfectly Correct (1.0)
    Explanation: 
The task asks for the capital of the United States. The candidate response states that the capital is Washington, D.C. (District of Columbia), which is factually correct. The response also provides additional accurate information about its location along the Potomac River and mentions key government buildings like the White House, Capitol Building, and Supreme Court. No tools were called in the context for this question, so we evaluate based on factual accuracy alone. The answer is accurate and complete.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5be27ec18f7a132ae8f73f317cb6'}}

    

### Tool selection accuracy and parameter selection accuracy

Let's now evaluate our agent for the tool and parameter selection. Both metrics are evaluated at the span level.

In [9]:
parameter_results = eval_client.run(
    evaluator_ids=["Builtin.ToolParameterAccuracy", "Builtin.ToolSelectionAccuracy"],
    agent_id=agent_id_langgraph,
    session_id=session_id_langgraph,
    look_back_time=timedelta(hours=24),
)

Let's now analyze the results. In this case, we are evaluating the session with two different metrics in the same run. That means that we now need to know which evaluator is producing each response. We can do that with the `evaluator_name` property of the result. Let's see how well our agent used tools:

In [10]:
for result in parameter_results:
    information = f"""
    Metric: {result.get("evaluatorId", "")}
    Value: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))
    print("================================================")


    Metric: Builtin.ToolParameterAccuracy
    Value: Yes (1.0)
    Explanation: 
The tool-call uses the 'calculator' tool with a single parameter 'expression' set to '2+2'. 

Analyzing the parameter:
- Parameter name: 'expression'
- Parameter value: '2+2'
- Source: The user explicitly asked "What is 2+2?" in the conversation history

The parameter value '2+2' is directly extracted from the user's question. The user literally asked about the mathematical expression "2+2", and this exact expression was passed to the calculator tool. This is a faithful representation of what the user requested.

The parameter format appears correct - it's a string representation of a mathematical expression, which is the expected format for a calculator tool's expression parameter.

There is no fabrication or hallucination here. The AI assistant simply took the mathematical expression from the user's question and passed it as a parameter to the calculator tool. This is a straightforward, faithful use of information from the preceding context.

All parameter values are directly traceable to the user's input, with no invented or assumed values.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bd572d2137536aa4bf4740da4d3', 'spanId': '32fd62ba5af7a9cd'}}

    


    Metric: Builtin.ToolParameterAccuracy
    Value: Yes (1.0)
    Explanation: 
The user asked 'What is the weather now?' which is a straightforward request for current weather information. The tool-call made is to a 'weather' function with empty parameters: `{'parameters': {}}`. 

Since no API schema is provided in the 'Available tool-calls' section, I cannot verify what parameters are expected, optional, or required for the weather tool. However, I can evaluate based on the information available:

1. The user did not specify a location in their query 'What is the weather now?'
2. The user did not specify any time parameters (though 'now' implies current time)
3. The parameters object is empty `{}`

The key question is whether omitting all parameters (particularly location) constitutes faithful use of the preceding context. The user said 'now' which implies current time, but provided no location information. 

Without the API schema, I cannot determine if location is a required parameter or if the API has a default behavior (e.g., using IP-based location detection). If location were a required parameter according to the schema, then omitting it would be unfaithful to the schema requirements.

However, given that the tool-call executed successfully and returned 'sunny' without error, this suggests the empty parameters were acceptable to the API. The assistant did not fabricate any parameter values - it simply passed no parameters, which appears to be valid for this API.

The parameters used (none) are faithful to what the user provided (no specific location or other details).

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bdf7584841d6f9de7aa63444712', 'spanId': '68a9825679b080dd'}}

    


    Metric: Builtin.ToolSelectionAccuracy
    Value: No (0.0)
    Explanation: 
The user asks 'What is 2+2?' which is a straightforward arithmetic question. While this is an extremely simple calculation that most AI assistants could answer directly without tools, using a calculator tool is technically a valid approach to ensure accuracy. The action parameters are correct (expression: '2+2') and the tool returns the correct result ('4'). However, this represents an unnecessary use of computational resources for a trivial calculation that the assistant should be able to answer immediately from its training. A helpful assistant would typically reserve tool usage for more complex calculations or when precision beyond basic arithmetic is required. For such a simple addition problem that any elementary school student can solve mentally, calling an external calculator tool is overkill and inefficient. The action does technically address the user's request and provides the correct answer, but it's not the most reasonable approach. A direct response would be faster, more efficient, and equally accurate. That said, the action does successfully fulfill the user's need for an answer, even if the method is unnecessarily complex.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bd572d2137536aa4bf4740da4d3', 'spanId': '32fd62ba5af7a9cd'}}

    


    Metric: Builtin.ToolSelectionAccuracy
    Value: Yes (1.0)
    Explanation: 
The user asked 'What is the weather now?' which is a direct request for current weather information. The assistant called the 'weather' tool to retrieve this information, which directly addresses the user's question. This is a straightforward and appropriate response to the user's request.

The tool call is justified because:
1. It directly addresses the user's explicit request for weather information
2. It is clearly aligned with the user's intent to know the current weather conditions
3. While the parameters are empty (which might be suboptimal as location information could make the response more useful), the tool still returns a result ('sunny'), suggesting it has default behavior that provides useful information
4. A helpful assistant would reasonably call a weather tool when asked about current weather conditions

The fact that the tool successfully returned 'sunny' indicates that even without explicit parameters, the tool was able to provide meaningful weather information, likely using default location settings or context. This makes the action useful and appropriate for serving the user's needs.

The action is a reasonable and direct response to fulfill the user's request for weather information.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bdf7584841d6f9de7aa63444712', 'spanId': '68a9825679b080dd'}}

    

### Using custom evaluator

Now that we have used evaluators in the session, trace and span level, let's use our custom metric to evaluate our response quality:

In [11]:
custom_results = eval_client.run(
    evaluator_ids=[evaluator_id],
    agent_id=agent_id_langgraph,
    session_id=session_id_langgraph,
    look_back_time=timedelta(hours=24),
)

Let's now take a look at the evaluation results. In this case, we are evaluating an agent that has the following instructions:

```
You're a helpful assistant. You can do simple math calculation, and tell the weather.
```

For our evaluation metric we are penalizing the agent for going out of scope with a `Very Poor` quality as stated in our evaluation instructions:

```
...
**IMPORTANT**: A response quality can only be high if the agent remains in its original scope. Penalize agents that answer questions outside its original scope with a Very Poor classification.
...
```

Since we are evaluating the following questions:

* What is the weather now?
* How much is 2+2?
* Can you tell me the capital of the US?

We expect the agent to have a `Very Poor` evaluation for the last question.

In [12]:
for result in custom_results:
    information = f"""
    Metric: {result.get("evaluatorId", "")}
    Value: {result.get("label", result.get("rating", "N/A"))} ({result.get("value", result.get("score", "N/A"))})
    Explanation: \n{result.get("explanation", result.get("reason", ""))}\n
    Token Usage: {result.get("token_usage", {})}\n
    Context: {result.get("context", {})}\n
    """
    display(Markdown(information))
    print("================================================")


    Metric: response_quality_for_scope_b40a1c32-vBGSJi4Nz4
    Value: Very Good (1.0)
    Explanation: 
The task asks to evaluate whether the Assistant's response is good and accurate for a mathematical query. The user asked 'What is 2+2?' which falls within the Assistant's scope (weather and mathematical queries). The Assistant used a calculator tool which correctly returned '4', and the Assistant's response states '2 + 2 = **4**'. This is mathematically accurate and directly answers the question. The response stays within the allowed scope (math question), provides the correct calculation, and presents it clearly. There are no errors, omissions, or issues with the factual content. The formatting with bold text doesn't detract from the correctness of the answer. This is a complete and accurate response to the mathematical query.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bd572d2137536aa4bf4740da4d3'}}

    


    Metric: response_quality_for_scope_b40a1c32-vBGSJi4Nz4
    Value: Very Good (1.0)
    Explanation: 
The assistant is designed to answer questions about weather and mathematical queries only. In the context provided, the user asks 'What is the weather now?' which falls within the assistant's scope. The weather tool was called and returned 'sunny' as the result. The assistant's response states 'The weather right now is **sunny**! It's a nice day out. ☀️' This response is accurate as it correctly conveys the weather information obtained from the tool (sunny). The assistant stays within its designated scope by answering a weather-related question. The addition of 'It's a nice day out' with a sun emoji is a reasonable interpretation of sunny weather and adds helpful context without introducing inaccuracy. The response directly and correctly answers the user's question using the tool result provided. There are no factual errors, the information is complete, and the assistant has not deviated from its allowed scope of weather and math questions.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5bdf7584841d6f9de7aa63444712'}}

    


    Metric: response_quality_for_scope_b40a1c32-vBGSJi4Nz4
    Value: Very Poor (0.0)
    Explanation: 
The evaluation criteria explicitly states that the agent should ONLY answer questions about weather and mathematical queries. The instructions emphasize that responses outside this original scope should be penalized with a 'Very Poor' classification regardless of factual accuracy.

Looking at the context, the agent correctly handled the first two queries: a math question (2+2) and a weather question. However, when asked 'Can you tell me the capital of the US?', the agent provided a detailed response about Washington, D.C.

While the response is factually accurate (Washington, D.C. is indeed the capital of the United States, and the details provided about its location and government buildings are correct), this question falls outside the agent's defined scope of weather and math queries. The question about U.S. geography/civics is neither a weather inquiry nor a mathematical calculation.

According to the evaluation instructions, answering questions outside the original scope warrants a very poor classification, regardless of how accurate the content is. The agent violated its operational boundaries by providing information on a topic it should have declined to answer or redirected back to its allowed domains.

    Token Usage: {}

    Context: {'spanContext': {'sessionId': '722e855f-5939-4aad-8afd-bc3d1a8a9ec6', 'traceId': '69fa5be27ec18f7a132ae8f73f317cb6'}}

    

### Saving evaluation results

You can save the evaluation results to a file for later analysis. Serialize the results list directly to JSON.

In [13]:
os.makedirs("evals_results", exist_ok=True)
save_results = eval_client.run(
    evaluator_ids=[evaluator_id],
    agent_id=agent_id_langgraph,
    session_id=session_id_langgraph,
    look_back_time=timedelta(hours=24),
)

with open("evals_results/output.json", "w") as f:
    json.dump(save_results, f, indent=2, default=str)
print(f"Saved {len(save_results)} evaluation result(s) to evals_results/output.json")

Saved 3 evaluation result(s) to evals_results/output.json


### Congrats!

You have now evaluated your agent with the on-demand capabilities. In the next tutorial, we will automate the evaluation of the agent for a production environment by setting an online evaluator and connecting it with the agent.